Name & ID : Muhammad Haziq bin Abdullah (SW01083756), Mohamad Naqib bin Mustapa (SW01083743)


In [17]:
import pandas as pd
import re
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [18]:
df = pd.read_csv("Reviews.csv")

In [19]:
df = df[['Score', 'Text']].dropna()

In [20]:
df = df[df['Score'] != 3]

In [21]:
df['Label'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [22]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\\S+|www\\S+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

df['CleanText'] = df['Text'].apply(preprocess_text)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    df['CleanText'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)

In [24]:
# -----------------------------
# 1. Lexicon-Based Method
# -----------------------------
lexicon_pred = []
for text in X_test:
    polarity = TextBlob(text).sentiment.polarity
    lexicon_pred.append(1 if polarity >= 0 else 0)

print("Lexicon-Based Method")
print("Accuracy:", accuracy_score(y_test, lexicon_pred))
print(classification_report(y_test, lexicon_pred))

Lexicon-Based Method
Accuracy: 0.8613105369759326
              precision    recall  f1-score   support

           0       0.59      0.37      0.45     16407
           1       0.89      0.95      0.92     88756

    accuracy                           0.86    105163
   macro avg       0.74      0.66      0.69    105163
weighted avg       0.84      0.86      0.85    105163



In [27]:
# -----------------------------
# 2. Bag-of-Words + Linear SVM
# -----------------------------
bow_vectorizer = CountVectorizer(
    stop_words='english',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    max_features=5000
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

bow_model = LinearSVC()
bow_model.fit(X_train_bow, y_train)
bow_pred = bow_model.predict(X_test_bow)

print("Bag-of-Words + Linear SVM")
print("Accuracy:", accuracy_score(y_test, bow_pred))
print(classification_report(y_test, bow_pred))

Bag-of-Words + Linear SVM
Accuracy: 0.9264570238582011
              precision    recall  f1-score   support

           0       0.84      0.66      0.74     16407
           1       0.94      0.98      0.96     88756

    accuracy                           0.93    105163
   macro avg       0.89      0.82      0.85    105163
weighted avg       0.92      0.93      0.92    105163



In [26]:
# -----------------------------
# 3. TF-IDF + Linear SVM
# -----------------------------
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    max_features=5000
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_model = LinearSVC()
tfidf_model.fit(X_train_tfidf, y_train)
tfidf_pred = tfidf_model.predict(X_test_tfidf)

print("TF-IDF + Linear SVM")
print("Accuracy:", accuracy_score(y_test, tfidf_pred))
print(classification_report(y_test, tfidf_pred))

TF-IDF + Linear SVM
Accuracy: 0.9306219868204597
              precision    recall  f1-score   support

           0       0.83      0.70      0.76     16407
           1       0.95      0.97      0.96     88756

    accuracy                           0.93    105163
   macro avg       0.89      0.84      0.86    105163
weighted avg       0.93      0.93      0.93    105163



(1)The lexicon-based method had the weakest performance because it depends only on predefined word polarity and struggles with context, sarcasm, product-specific language, and mixed opinions.
(2)The BoW + Linear SVM model performed much better because it learned patterns from the dataset directly.
(3)The TF-IDF + Linear SVM model achieved the best result because TF-IDF reduced the impact of overly common words and highlighted more meaningful terms for sentiment classification.

Among the three methods, TF-IDF + Linear SVM is the best model for sentiment classification on Reviews.csv because it achieved the highest accuracy and the most balanced overall performance. The lexicon-based approach works only as a basic baseline, while BoW + Linear SVM is strong but slightly weaker than TF-IDF. Therefore, the most suitable method for this dataset is TF-IDF with a machine learning classifier.